# Qwen2 validation and GIS export

This notebook builds on the classification and calibration outputs from Notebook 2. It has two purposes:

1. Rebuild the calibrated GIS-ready files from the article-level scores;
2. Evaluate the manual validation sample by comparing manual scores with both calibrated and raw Qwen2 scores.

Notebook 2 produces the article-level Qwen2 classifications, the benchmark-calibrated article scores, and the manual validation sample. This notebook starts from those saved files (and is not rerunning Qwen2).

The notebook:

- loads `article_news_values_calibrated_to_harcup_oneill.csv`;
- loads `locations_final.csv` and `article_id_mapping.csv`;
- rebuilds `location_mentions_with_calibrated_news_values_for_gis.csv`;
- rebuilds `location_news_value_summary_by_location_calibrated.csv`;
- calculates validation tables after the manual validation columns have been filled in.


## 1. Imports and settings

This section loads the packages and defines the shared settings used for GIS export and validation.


In [ ]:
# Import used packages
from pathlib import Path
from typing import Iterable
import numpy as np
import pandas as pd

# Display wider tables to make validation summaries easier to inspect in the notebook.
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 180)


In [ ]:
#Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


In [ ]:
# Use the same processed-data folder as Notebooks 1 and 2.
DATA_DIR = Path("/content/drive/MyDrive/Thesis/data_processed")

# Define the six news values used throughout the classification, GIS, and validation workflow.
NEWS_VALUE_NAMES = [
    "entertainment",
    "bad_news",
    "magnitude",
    "good_news",
    "celebrity",
    "power_elite",
]

# Keep score-column names stable because downstream CSV and GIS imports depend on them.
SCORE_COLS = [f"{name}_score" for name in NEWS_VALUE_NAMES]
RAW_SCORE_COLS = [f"raw_{name}_score" for name in NEWS_VALUE_NAMES]
MANUAL_SCORE_COLS = [f"manual_{name}_score" for name in NEWS_VALUE_NAMES]
ALLOWED_SCORES = {0, 50, 100}

# Use broad European Netherlands coordinate bounds to remove obvious geocoding errors.
NL_LAT_MIN, NL_LAT_MAX = 50.6, 53.7
NL_LON_MIN, NL_LON_MAX = 3.2, 7.3


## 2. Helper functions

In [ ]:
# Resolve input and output paths within the processed-data folder.
def input_path(filename: str, required: bool = True) -> Path | None:
    path = DATA_DIR / filename

    if path.exists():
        return path

    if required:
        raise FileNotFoundError(f"Could not find {filename} in {DATA_DIR}")

    return None

def output_path(filename: str) -> Path:
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    return DATA_DIR / filename


# Fail early if a required column is missing.
def require_columns(df: pd.DataFrame, required: Iterable[str], frame_name: str) -> None:
    missing = [col for col in required if col not in df.columns]
    if missing:
        raise ValueError(f"{frame_name} is missing required columns: {missing}")


# Coerce score columns to the allowed values: 0, 50, or 100.
def coerce_score(series: pd.Series) -> pd.Series:
    numeric = pd.to_numeric(series, errors="coerce")
    return numeric.where(numeric.isin(list(ALLOWED_SCORES)))


# Standardise article IDs before merging tables from different stages of the workflow.
def clean_id(series: pd.Series) -> pd.Series:
    return series.astype(str).str.strip()


## 3. Load existing generated CSVs

This section loads the saved outputs from the previous notebooks. Essential files are required explicitly. If one is missing, the notebook stops.

The raw article-level classification file is used only when raw score columns are not already present in the calibrated article file.


In [ ]:
# Locate the input files needed for GIS rebuilding and validation.
LOCATIONS_PATH = input_path("locations_final.csv", required=True)
ARTICLE_ID_MAPPING_PATH = input_path("article_id_mapping.csv", required=True)
CALIBRATED_ARTICLE_VALUES_PATH = input_path("article_news_values_calibrated_to_harcup_oneill.csv", required=True)
RAW_ARTICLE_VALUES_PATH = input_path("article_news_values.csv", required=False)
MANUAL_VALIDATION_SAMPLE_PATH = input_path("manual_validation_sample.csv", required=False)

# Print resolved paths to document which saved files are being used.
print("Locations file:", LOCATIONS_PATH)
print("Article ID mapping file:", ARTICLE_ID_MAPPING_PATH)
print("Calibrated article values file:", CALIBRATED_ARTICLE_VALUES_PATH)
print("Raw article values file:", RAW_ARTICLE_VALUES_PATH)
print("Manual validation sample file:", MANUAL_VALIDATION_SAMPLE_PATH)


Locations file: /content/drive/MyDrive/Thesis/data_processed/locations_final.csv
Article ID mapping file: /content/drive/MyDrive/Thesis/data_processed/article_id_mapping.csv
Calibrated article values file: /content/drive/MyDrive/Thesis/data_processed/article_news_values_calibrated_to_harcup_oneill.csv
Raw article values file: /content/drive/MyDrive/Thesis/data_processed/article_news_values.csv
Manual validation sample file: /content/drive/MyDrive/Thesis/data_processed/manual_validation_sample.csv


In [ ]:
# Load the calibrated article scores, location mentions, and article-ID mapping.
calibrated_articles = pd.read_csv(CALIBRATED_ARTICLE_VALUES_PATH)
locations_raw = pd.read_csv(LOCATIONS_PATH)
article_id_mapping = pd.read_csv(ARTICLE_ID_MAPPING_PATH)

# Verify required columns before performing merges.
require_columns(calibrated_articles, ["article_id", *SCORE_COLS], "calibrated_articles")
require_columns(locations_raw, ["article_id", "location", "lat", "lon"], "locations_raw")
require_columns(article_id_mapping, ["article_id", "article_id_original"], "article_id_mapping")

# Recover raw Qwen2 score columns from `article_news_values.csv` if they are absent from the calibrated file.
missing_raw_cols = [col for col in RAW_SCORE_COLS if col not in calibrated_articles.columns]
if missing_raw_cols:
    if RAW_ARTICLE_VALUES_PATH is None:
        raise ValueError("Raw score columns are missing and article_news_values.csv was not found.")
    raw_articles = pd.read_csv(RAW_ARTICLE_VALUES_PATH)
    require_columns(raw_articles, ["article_id", *SCORE_COLS], "raw_articles")
    raw_articles = raw_articles.rename(columns={col: f"raw_{col}" for col in SCORE_COLS})
    calibrated_articles = calibrated_articles.merge(
        raw_articles[["article_id", *RAW_SCORE_COLS]].drop_duplicates("article_id"),
        on="article_id",
        how="left",
    )

# Coerce all model score columns to the permitted values: 0, 50, or 100.
for col in SCORE_COLS + RAW_SCORE_COLS:
    calibrated_articles[col] = coerce_score(calibrated_articles[col]).astype("Int64")

print("Calibrated article rows:", len(calibrated_articles))
print("Raw location rows:", len(locations_raw))
print("Mapping rows:", len(article_id_mapping))
display(calibrated_articles.head())


Calibrated article rows: 12225
Raw location rows: 19802
Mapping rows: 55772


,entertainment_score,bad_news_score,magnitude_score,good_news_score,celebrity_score,power_elite_score,dominant_news_values,dominant_news_values_json,short_reason,parse_ok,parse_error,article_id,title,date,subjects,classification_text_char_count,body_char_limit,model_name,prompt_type,fewshot_examples_used,raw_entertainment_score,raw_bad_news_score,raw_magnitude_score,raw_good_news_score,raw_celebrity_score,raw_power_elite_score
0,0,100,50,0,0,50,bad_news,"[""bad_news""]","Health alert about hepatitis A virus infection; high magnitude due widespread potential spread, low entertainment/newsworthiness, high bad news score",True,NaN,436,12 mensen (erg) ziek: dit weten we over het hepatitisvirus in AH-bessen,14 januari 2025 dinsdag 09:52 AM GMT,Government Departments + Authorities; Hepatitis; Viruses; Health Care Regulation + Policy; Infectious Disease; Public Health; Health Care Professionals; Food Inspection; Food +...,1211,750,Qwen/Qwen2-1.5B-Instruct,few-shot,4,0,100,50,0,0,50
1,0,100,0,0,0,50,bad_news,"[""bad_news""]",Infectious disease outbreak; high magnitude; low entertainment/celebrity presence; low news value,True,NaN,438,"12 zieken, 2 opnames: dit weten we over het hepatitisvirus in AH-bessen",14 januari 2025 dinsdag 09:52 AM GMT,Hepatitis; Infectious Disease; Viruses; Government Departments + Authorities; Health Care Regulation + Policy; Public Health; Food Inspection; Food + Beverage Regulation + Poli...,1211,750,Qwen/Qwen2-1.5B-Instruct,few-shot,4,0,100,50,0,0,50
2,0,100,50,0,0,50,bad_news,"[""bad_news""]","Fraudulent webshop exposed by police; high-profile institution central, magnitude secondary due widespread exposure, negative consequences",True,NaN,443,125.000 mensen bezochten nepwebshop waarmee politie waarschuwt voor fraude,25 januari 2025 zaterdag 09:06 AM GMT,Fraud + Financial Crime; Mail Order + Internet Retailing; Police Forces; Education + Training; Education Systems + Institutions; Electronic Commerce; Law Enforcement,1149,750,Qwen/Qwen2-1.5B-Instruct,few-shot,4,0,100,50,0,0,50
3,0,50,100,0,0,0,magnitude,"[""magnitude""]","Fraudulent website targeted by police; high-profile event central, magnitude secondary due widespread awareness, low/no celebrity involvement, low/no power elite involvement",True,NaN,444,125.000 mensen bezoeken nepwebshop waarmee politie waarschuwt voor fraude,25 januari 2025 zaterdag 09:06 AM GMT,Mail Order + Internet Retailing; Fraud + Financial Crime; Police Forces; Education + Training; Education Systems + Institutions; Electronic Commerce; Law Enforcement,1148,750,Qwen/Qwen2-1.5B-Instruct,few-shot,4,0,0,100,0,0,0
4,0,100,50,0,0,50,bad_news,"[""bad_news""]","Police raids on multiple explosive incidents; serious harm central, magnitude secondary due high number of arrests, wide scope of locations",True,NaN,468,"15 arrestaties voor meerdere explosies in Rotterdam, Zandvoort en Den Haag",20 januari 2025 maandag 03:02 PM GMT,Arrests; Police Forces; Misdemeanors; Criminal Offenses; Weapons + Arms,1127,750,Qwen/Qwen2-1.5B-Instruct,few-shot,4,0,100,50,0,0,50


## 4. Rebuild the calibrated GIS mention file

This section creates `location_mentions_with_calibrated_news_values_for_gis.csv`.

The file has one row per article-location mention. Each mention receives the calibrated article-level news-value scores from Notebook 2, while raw Qwen2 scores are retained when available.

It still has to match `locations_final.csv`, which still contains original article IDs, to the numeric article IDs used in the Qwen2 output. I used ChatGPT as coding support to implement and check this matching logic.


In [ ]:
# Map original location-table article IDs to the numeric article IDs used in the Qwen2 output.
def standardize_location_article_ids(locations: pd.DataFrame, mapping: pd.DataFrame) -> pd.DataFrame:
    # The location file still uses the original article ID, so rename it before merging.
    loc = locations.rename(columns={"article_id": "article_id_original"}).copy()
    map_df = mapping.copy()

    # Join on original ID plus available article metadata to avoid accidental matches between similar records.
    key_candidates = ["article_id_original", "file_name", "title", "date"]
    key_cols = [col for col in key_candidates if col in loc.columns and col in map_df.columns]
    if "article_id_original" not in key_cols:
        raise ValueError("Both locations_final.csv and article_id_mapping.csv must contain article_id_original.")

    # Normalise join keys before merging.
    for col in key_cols:
        loc[col] = clean_id(loc[col])
        map_df[col] = clean_id(map_df[col])

    map_keep = ["article_id", *key_cols]
    # Merge the numeric article ID onto each location row.
    loc = loc.merge(
        map_df[map_keep].drop_duplicates(key_cols),
        on=key_cols,
        how="left",
    )

    # Stop if any location rows cannot be linked to a numeric article ID.
    missing = int(loc["article_id"].isna().sum())
    if missing:
        raise ValueError(f"{missing} location rows could not be matched to numeric article IDs.")

    loc["article_id"] = loc["article_id"].astype(int)
    ordered = ["article_id", "article_id_original"] + [col for col in loc.columns if col not in ["article_id", "article_id_original"]]
    return loc[ordered]


# Filter only obvious non-European-Netherlands coordinates.
def filter_european_netherlands(locations: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    loc = locations.copy()
    loc["lat"] = pd.to_numeric(loc["lat"], errors="coerce")
    loc["lon"] = pd.to_numeric(loc["lon"], errors="coerce")

    in_bbox = (
        loc["lat"].between(NL_LAT_MIN, NL_LAT_MAX)
        & loc["lon"].between(NL_LON_MIN, NL_LON_MAX)
    )

    return loc.loc[in_bbox].copy(), loc.loc[~in_bbox].copy()


# Merge calibrated and raw article-level scores onto each article-location mention.
def rebuild_calibrated_gis_mentions() -> pd.DataFrame:
    # First convert original article IDs to numeric article IDs.
    locations_numeric = standardize_location_article_ids(locations_raw, article_id_mapping)
    locations_nl, excluded_locations = filter_european_netherlands(locations_numeric)

    # Remove exact duplicate article-location-coordinate rows.
    duplicate_subset = ["article_id", "location", "lat", "lon"]
    duplicate_rows = locations_nl[locations_nl.duplicated(subset=duplicate_subset, keep=False)].copy()
    locations_nl = locations_nl.drop_duplicates(subset=duplicate_subset, keep="first").copy()

    # Save diagnostic files for excluded coordinates and duplicate article-location rows.
    excluded_locations.to_csv(output_path("diagnostic_excluded_non_netherlands_location_rows.csv"), index=False, encoding="utf-8-sig")
    duplicate_rows.to_csv(output_path("diagnostic_duplicate_article_location_rows.csv"), index=False, encoding="utf-8-sig")

    article_scores = calibrated_articles.drop_duplicates("article_id", keep="last").copy()
    article_scores["article_id"] = article_scores["article_id"].astype(int)

   # Preserve model metadata where available for auditability.
    metadata_cols = [
        "dominant_news_values",
        "dominant_news_values_json",
        "short_reason",
        "parse_ok",
        "parse_error",
        "model_name",
        "body_char_limit",
        "prompt_type",
        "fewshot_examples_used",
    ]
    merge_cols = ["article_id", *SCORE_COLS, *RAW_SCORE_COLS]
    merge_cols += [col for col in metadata_cols if col in article_scores.columns]

    gis = locations_nl.merge(article_scores[merge_cols], on="article_id", how="left")
    gis["classification_missing"] = gis[SCORE_COLS[0]].isna()

    # Convert score columns back to clean integer-like values.
    for col in SCORE_COLS + RAW_SCORE_COLS:
        gis[col] = coerce_score(gis[col]).astype("Int64")

    # Save the rebuilt GIS mention file under the calibrated GIS filename.
    gis_path = output_path("location_mentions_with_calibrated_news_values_for_gis.csv")
    gis.to_csv(gis_path, index=False, encoding="utf-8-sig")

    print("Saved:", gis_path)
    print("Rows before NL filter:", len(locations_numeric))
    print("Rows excluded outside European NL bbox:", len(excluded_locations))
    print("Duplicate article-location-coordinate rows removed:", len(duplicate_rows))
    print("Final GIS mention rows:", len(gis))
    print("Rows missing classification:", int(gis["classification_missing"].sum()))
    return gis

final_gis_df = rebuild_calibrated_gis_mentions()
display(final_gis_df.head())


Saved: /content/drive/MyDrive/Thesis/data_processed/location_mentions_with_calibrated_news_values_for_gis.csv
Rows before NL filter: 19802
Rows excluded outside European NL bbox: 28
Duplicate article-location-coordinate rows removed: 0
Final GIS mention rows: 19774
Rows missing classification: 0


,article_id,article_id_original,file,file_name,title,date,subjects,location,lat,lon,entertainment_score,bad_news_score,magnitude_score,good_news_score,celebrity_score,power_elite_score,raw_entertainment_score,raw_bad_news_score,raw_magnitude_score,raw_good_news_score,raw_celebrity_score,raw_power_elite_score,dominant_news_values,dominant_news_values_json,short_reason,parse_ok,parse_error,model_name,body_char_limit,prompt_type,fewshot_examples_used,classification_missing
0,436,12 mensen (erg) ziek_ dit weten we over het hepatitisvirus in AH-bessen,/content/drive/MyDrive/Thesis/data/01-2025/12 mensen (erg) ziek_ dit weten we over het hepatitisvirus in AH-bessen.DOCX,12 mensen (erg) ziek_ dit weten we over het hepatitisvirus in AH-bessen.DOCX,12 mensen (erg) ziek: dit weten we over het hepatitisvirus in AH-bessen,14 januari 2025 dinsdag 09:52 AM GMT,Government Departments + Authorities; Hepatitis; Viruses; Health Care Regulation + Policy; Infectious Disease; Public Health; Health Care Professionals; Food Inspection; Food +...,Wageningen,51.966302,5.666281,0,100,50,0,0,50,0,100,50,0,0,50,bad_news,"[""bad_news""]","Health alert about hepatitis A virus infection; high magnitude due widespread potential spread, low entertainment/newsworthiness, high bad news score",True,NaN,Qwen/Qwen2-1.5B-Instruct,750,few-shot,4,False
1,438,"12 zieken, 2 opnames_ dit weten we over het hepatitisvirus in AH-bessen","/content/drive/MyDrive/Thesis/data/01-2025/12 zieken, 2 opnames_ dit weten we over het hepatitisvirus in AH-bessen.DOCX","12 zieken, 2 opnames_ dit weten we over het hepatitisvirus in AH-bessen.DOCX","12 zieken, 2 opnames: dit weten we over het hepatitisvirus in AH-bessen",14 januari 2025 dinsdag 09:52 AM GMT,Hepatitis; Infectious Disease; Viruses; Government Departments + Authorities; Health Care Regulation + Policy; Public Health; Food Inspection; Food + Beverage Regulation + Poli...,Wageningen,51.966302,5.666281,0,100,0,0,0,50,0,100,50,0,0,50,bad_news,"[""bad_news""]",Infectious disease outbreak; high magnitude; low entertainment/celebrity presence; low news value,True,NaN,Qwen/Qwen2-1.5B-Instruct,750,few-shot,4,False
2,443,125.000 mensen bezochten nepwebshop waarmee politie waarschuwt voor fraude,/content/drive/MyDrive/Thesis/data/01-2025/125.000 mensen bezochten nepwebshop waarmee politie waarschuwt voor fraude.DOCX,125.000 mensen bezochten nepwebshop waarmee politie waarschuwt voor fraude.DOCX,125.000 mensen bezochten nepwebshop waarmee politie waarschuwt voor fraude,25 januari 2025 zaterdag 09:06 AM GMT,Fraud + Financial Crime; Mail Order + Internet Retailing; Police Forces; Education + Training; Education Systems + Institutions; Electronic Commerce; Law Enforcement,Heemskerk,52.510433,4.672354,0,100,50,0,0,50,0,100,50,0,0,50,bad_news,"[""bad_news""]","Fraudulent webshop exposed by police; high-profile institution central, magnitude secondary due widespread exposure, negative consequences",True,NaN,Qwen/Qwen2-1.5B-Instruct,750,few-shot,4,False
3,444,125.000 mensen bezoeken nepwebshop waarmee politie waarschuwt voor fraude,/content/drive/MyDrive/Thesis/data/01-2025/125.000 mensen bezoeken nepwebshop waarmee politie waarschuwt voor fraude.DOCX,125.000 mensen bezoeken nepwebshop waarmee politie waarschuwt voor fraude.DOCX,125.000 mensen bezoeken nepwebshop waarmee politie waarschuwt voor fraude,25 januari 2025 zaterdag 09:06 AM GMT,Mail Order + Internet Retailing; Fraud + Financial Crime; Police Forces; Education + Training; Education Systems + Institutions; Electronic Commerce; Law Enforcement,Heemskerk,52.510433,4.672354,0,50,100,0,0,0,0,0,100,0,0,0,magnitude,"[""magnitude""]","Fraudulent website targeted by police; high-profile event central, magnitude secondary due widespread awareness, low/no celebrity involvement, low/no power elite involvement",True,NaN,Qwen/Qwen2-1.5B-Instruct,750,few-shot,4,False
4,468,"15 arrestaties voor meerdere explosies in Rotterdam, Zandvoort en Den Haag","/content/drive/MyDrive/Thesis/data/01-2025/15 ar

### Check the GIS mention file

This check documents whether the rebuilt GIS mention table contains valid coordinates, linked article IDs, and permitted news-value scores.


In [ ]:
# Summarise row counts, missing classifications, and coordinate coverage for the rebuilt GIS table.
checks = {
    "rows": len(final_gis_df),
    "unique_articles": final_gis_df["article_id"].nunique(),
    "unique_location_points": final_gis_df[["location", "lat", "lon"]].drop_duplicates().shape[0],
    "missing_lat_lon_rows": int(final_gis_df[["lat", "lon"]].isna().any(axis=1).sum()),
    "rows_outside_european_nl_bbox": int((
        ~final_gis_df["lat"].between(NL_LAT_MIN, NL_LAT_MAX)
        | ~final_gis_df["lon"].between(NL_LON_MIN, NL_LON_MAX)
    ).sum()),
    "rows_missing_classification": int(final_gis_df["classification_missing"].sum()),
}

display(pd.DataFrame([checks]))

# Report the distinct score values present in each calibrated and raw score column.
score_value_check = []
for col in SCORE_COLS + RAW_SCORE_COLS:
    score_value_check.append({
        "column": col,
        "values_present": sorted(final_gis_df[col].dropna().astype(int).unique().tolist()),
    })
display(pd.DataFrame(score_value_check))


,rows,unique_articles,unique_location_points,missing_lat_lon_rows,rows_outside_european_nl_bbox,rows_missing_classification
0,19774,12225,880,0,0,0


,column,values_present
0,entertainment_score,"[0, 50, 100]"
1,bad_news_score,"[0, 50, 100]"
2,magnitude_score,"[0, 50, 100]"
3,good_news_score,"[0, 50, 100]"
4,celebrity_score,"[0, 50, 100]"
5,power_elite_score,"[0, 50, 100]"
6,raw_entertainment_score,"[0, 50, 100]"
7,raw_bad_news_score,"[0, 100]"
8,raw_magnitude_score,"[0, 50, 100]"
9,raw_good_news_score,"[0, 50, 100]"


## 5. Rebuild the calibrated by-location GIS summary

This section creates `location_news_value_summary_by_location_calibrated.csv`.

The mention-level GIS file has one row per article-location mention. This summary groups those rows into one row per unique location-coordinate pair and calculates:

- the number of article mentions;
- the mean calibrated score for each news value;
- the share of mentions where each news value is present;
- the dominant news-value category for mapping.

I used ChatGPT as coding support for the aggregation logic.


In [ ]:
# Identify the strongest news value for a location from mean calibrated scores.
def dominant_values_from_mean_scores(row: pd.Series, mean_cols: list[str], tie_threshold: float = 5.0) -> str:
    values = row[mean_cols].astype(float).sort_values(ascending=False)
    top_value = values.iloc[0]
    if pd.isna(top_value) or top_value <= 0:
        return "none"
    # Keep multiple dominant values when their mean scores are close to the maximum.
    top_cols = [col for col, value in values.items() if (top_value - value) < tie_threshold]
    return "; ".join(col.replace("_score_mean", "") for col in top_cols)


# Convert detailed dominant values into a simpler map category.
def dominant_map_category(dominant_values: str) -> str:
    if pd.isna(dominant_values) or dominant_values == "none":
        return "none"
    if ";" in str(dominant_values):
        return "mixed"
    return str(dominant_values)


# Aggregate mention-level rows into one row per location-coordinate pair.
def summarize_by_location(df: pd.DataFrame, score_cols: list[str], output_filename: str) -> pd.DataFrame:
    group_cols = ["location", "lat", "lon"]
    aggregation_spec = {"mention_count": ("article_id", "count")}

    # Calculate mention counts, mean scores, and presence/strength shares for each news value.
    for col in score_cols:
        aggregation_spec[f"{col}_mean"] = (col, "mean")
        aggregation_spec[f"{col}_present_share"] = (col, lambda series: (series > 0).mean())
        aggregation_spec[f"{col}_strong_share"] = (col, lambda series: (series == 100).mean())
        aggregation_spec[f"{col}_sum"] = (col, "sum")

    # Group all article-location rows into one row per location.
    summary = df.groupby(group_cols, dropna=False).agg(**aggregation_spec).reset_index()
    mean_cols = [f"{col}_mean" for col in score_cols]
    summary["dominant_news_values"] = summary.apply(lambda row: dominant_values_from_mean_scores(row, mean_cols), axis=1)
    summary["dominant_map_category"] = summary["dominant_news_values"].apply(dominant_map_category)

    # Save the by-location summary under the calibrated GIS summary filename.
    path = output_path(output_filename)
    summary.to_csv(path, index=False, encoding="utf-8-sig")
    print("Saved:", path)
    print("Rows:", len(summary))
    return summary

summary_by_location_calibrated = summarize_by_location(
    final_gis_df,
    SCORE_COLS,
    "location_news_value_summary_by_location_calibrated.csv",
)

display(summary_by_location_calibrated.sort_values("mention_count", ascending=False).head(20))


Saved: /content/drive/MyDrive/Thesis/data_processed/location_news_value_summary_by_location_calibrated.csv
Rows: 880


,location,lat,lon,mention_count,entertainment_score_mean,entertainment_score_present_share,entertainment_score_strong_share,entertainment_score_sum,bad_news_score_mean,bad_news_score_present_share,bad_news_score_strong_share,bad_news_score_sum,magnitude_score_mean,magnitude_score_present_share,magnitude_score_strong_share,magnitude_score_sum,good_news_score_mean,good_news_score_present_share,good_news_score_strong_share,good_news_score_sum,celebrity_score_mean,celebrity_score_present_share,celebrity_score_strong_share,celebrity_score_sum,power_elite_score_mean,power_elite_score_present_share,power_elite_score_strong_share,power_elite_score_sum,dominant_news_values,dominant_map_category
27,Amsterdam,52.373080,4.892453,2335,31.884368,0.472377,0.165310,74450,46.017131,0.640257,0.280086,107450,13.618844,0.231263,0.041113,31800,13.961456,0.170021,0.109208,32600,16.316916,0.273233,0.053105,38100,17.044968,0.312206,0.028694,39800,bad_news,bad_news
632,Rotterdam,51.924442,4.477750,1482,28.475034,0.426451,0.143050,42200,49.932524,0.656545,0.342105,74000,15.890688,0.260459,0.057355,23550,13.933873,0.169366,0.109312,20650,11.504723,0.201754,0.028340,17050,20.074224,0.365722,0.035762,29750,bad_news,bad_news
161,Den Haag,52.079984,4.311346,1452,14.256198,0.239669,0.045455,20700,45.592287,0.674931,0.236915,66200,12.224518,0.194904,0.049587,17750,5.681818,0.073691,0.039945,8250,6.198347,0.116391,0.007576,9000,18.904959,0.305096,0.073003,27450,bad_news,bad_news
727,Utrecht,52.090701,5.121563,842,21.793349,0.355107,0.080760,18350,40.795724,0.604513,0.211401,34350,15.67696,0.226841,0.086698,13200,11.817102,0.153207,0.083135,9950,7.779097,0.148456,0.007126,6550,11.045131,0.210214,0.010689,9300,bad_news,bad_news
279,Groningen,53.219065,6.568008,651,25.576037,0.411674,0.099846,16650,38.556068,0.557604,0.213518,25100,20.583717,0.274962,0.136713,13400,14.669739,0.173579,0.119816,9550,9.677419,0.179724,0.013825,6300,13.44086,0.238095,0.030722,8750,bad_news,bad_news
217,Eindhoven,51.439265,5.478633,610,33.442623,0.562295,0.106557,20400,39.098361,0.555738,0.226230,23850,14.918033,0.216393,0.081967,9100,21.065574,0.263934,0.157377,12850,13.52459,0.262295,0.008197,8250,17.04918,0.281967,0.059016,10400,bad_news,bad_news
316,Heerenveen,52.956825,5.927005,472,54.237288,0.836864,0.247881,25600,23.728814,0.372881,0.101695,11200,12.182203,0.169492,0.074153,5750,33.898305,0.413136,0.264831,16000,15.677966,0.307203,0.006356,7400,9.110169,0.141949,0.040254,4300,entertainment,entertainment
40,Arnhem,51.984257,5.910857,390,27.564103,0.430769,0.120513,10750,48.333333,0.658974,0.307692,18850,18.974359,0.284615,0.094872,7400,15.641026,0.194872,0.117949,6100,9.871795,0.176923,0.020513,3850,16.153846,0.302564,0.020513,6300,bad_news,bad_news
709,Tilburg,51.556240,5.088601,282,29.078014,0.450355,0.131206,8200,44.858156,0.609929,0.287234,12650,12.588652,0.216312,0.035461,3550,13.120567,0.173759,0.088652,3700,10.815603,0.209220,0.007092,3050,15.957447,0.294326,0.024823,4500,bad_news,bad_news
116,Breda,51.588785,4.776024,281,25.266904,0.427046,0.078292,7100,47.686833,0.626335,0.327402,13400,16.014235,0.263345,0.056940,4500,9.786477,0.142349,0.053381,2750,8.896797,0.177936,0.000000,2500,16.014235,0.302491,0.017794,4500,bad_news,bad_news


## 6. Manual validation: calibrated and raw

Notebook 2 created `manual_validation_sample.csv`. After the six `manual_*_score` columns have been filled in, this section compares manual scores with:

1. the calibrated Qwen2 scores used in the main thesis/GIS workflow;
2. the raw Qwen2 scores preserved for comparison.

The validation files are not required for GIS import, but they are important for evaluating how calibration changes agreement with manual coding. The results can show improvement, deterioration, or no meaningful difference depending on the news value and metric.


### Load the manual validation sample

This cell loads the validation sample and adds model score columns from the calibrated article file if they are not already present in the validation file.


In [ ]:
if MANUAL_VALIDATION_SAMPLE_PATH is None:
    raise FileNotFoundError("manual_validation_sample.csv was not found. Use the original notebook output or create the sample first.")

# Load the existing manual validation sample.
validation_sample = pd.read_csv(MANUAL_VALIDATION_SAMPLE_PATH)
print("Loaded validation sample:", MANUAL_VALIDATION_SAMPLE_PATH)
print("Rows:", len(validation_sample))

# Add calibrated and raw model score columns when they are missing from the validation file.
needed_from_article_scores = ["article_id", *SCORE_COLS, *RAW_SCORE_COLS]
missing_cols = [col for col in SCORE_COLS + RAW_SCORE_COLS if col not in validation_sample.columns]
if missing_cols:
    validation_sample = validation_sample.merge(
        calibrated_articles[needed_from_article_scores].drop_duplicates("article_id"),
        on="article_id",
        how="left",
    )

# Add empty manual score columns when the file has not yet been manually completed.
for col in MANUAL_SCORE_COLS:
    if col not in validation_sample.columns:
        validation_sample[col] = np.nan

display(validation_sample.head())


Loaded validation sample: /content/drive/MyDrive/Thesis/data_processed/manual_validation_sample.csv
Rows: 300


,article_id,classification_text,entertainment_score,bad_news_score,magnitude_score,good_news_score,celebrity_score,power_elite_score,manual_entertainment_score,manual_bad_news_score,manual_magnitude_score,manual_good_news_score,manual_celebrity_score,manual_power_elite_score,dominant_news_values,short_reason,prompt_type,fewshot_examples_used,dominant_news_values_json,parse_ok,parse_error,title_x,date_x,subjects_x,classification_text_char_count,body_char_limit,model_name,raw_entertainment_score,raw_bad_news_score,raw_magnitude_score,raw_good_news_score,raw_celebrity_score,raw_power_elite_score,title_y,date_y,subjects_y
0,36459,Article ID: 36459\nDate: 14 november 2025 vrijdag 07:41 AM GMT\nTitle: Ook 'lichte' bevingen in Groningen leiden tot relatief veel schade\nSubjects: Earthquakes; Natural Disast...,0,50,100,0,0,0,0,100,50,0,0,0,magnitude,"Earthquake causes significant damage; high magnitude central, low severity secondary, no celebrity or elite presence",few-shot,4,"[""magnitude""]",True,NaN,Ook 'lichte' bevingen in Groningen leiden tot relatief veel schade,14 november 2025 vrijdag 07:41 AM GMT,Earthquakes; Natural Disasters; Emergency Services,1115,750,Qwen/Qwen2-1.5B-Instruct,0,0,100,0,0,0,Ook 'lichte' bevingen in Groningen leiden tot relatief veel schade,14 november 2025 vrijdag 07:41 AM GMT,Earthquakes; Natural Disasters; Emergency Services
1,48187,Article ID: 48187\nDate: 1 januari 2025 woensdag 05:50 PM GMT\nTitle: Tweede dodelijk slachtoffer door vuurwerk: man in Tiel (46) overleden na explosie\nSubjects: Death + Dying...,0,100,50,0,0,50,0,100,50,0,0,50,bad_news,"Explosion kills two men; serious injuries central; police/court authority present but not main; magnitude secondary due fatalities, injuries, and severity of injuries",few-shot,4,"[""bad_news""]",True,NaN,Tweede dodelijk slachtoffer door vuurwerk: man in Tiel (46) overleden na explosie,1 januari 2025 woensdag 05:50 PM GMT,"Death + Dying; Police Forces; Wounds + Injuries; Health Care Facilities; Regional + Local Governments; Crime, Law Enforcement + Corrections; Criminal Law; Emergency Services; H...",1253,750,Qwen/Qwen2-1.5B-Instruct,0,100,50,0,0,50,Tweede dodelijk slachtoffer door vuurwerk: man in Tiel (46) overleden na explosie,1 januari 2025 woensdag 05:50 PM GMT,"Death + Dying; Police Forces; Wounds + Injuries; Health Care Facilities; Regional + Local Governments; Crime, Law Enforcement + Corrections; Criminal Law; Emergency Services; H..."
2,25926,Article ID: 25926\nDate: 28 augustus 2025 donderdag 09:52 AM GMT\nTitle: Live F1 | Verstappen staat pers te woord in aanloop naar GP van Nederland\nSubjects: Auto Racing; Tourn...,50,0,0,100,0,0,100,0,0,0,0,0,good_news,"Live blog about F1 race weekend in Zandvoort; entertainment central, high-level event secondary.",few-shot,4,"[""good_news""]",True,NaN,Live F1 | Verstappen staat pers te woord in aanloop naar GP van Nederland,28 augustus 2025 donderdag 09:52 AM GMT,Auto Racing; Tournaments; Auto Racing; Tournaments,664,750,Qwen/Qwen2-1.5B-Instruct,0,0,0,100,0,0,Live F1 | Verstappen staat pers te woord in aanloop naar GP van Nederland,28 augustus 2025 donderdag 09:52 AM GMT,Auto Racing; Tournaments; Auto Racing; Tournaments
3,17122,Article ID: 17122\nDate: 21 januari 2025 dinsdag 10:31 AM GMT\nTitle: Hospita in Enschede voor economisch daklozen\nSubjects: Social Security; Housing Market; Poverty + Homeles...,0,50,0,0,0,0,0,50,0,0,0,0,bad_news,NaN,few-shot,4,"[""bad_news""]",True,NaN,Hospita in Enschede voor economisch daklozen,21 januari 2025 dinsdag 10:31 AM GMT,Social Security; Housing Market; Poverty + Homelessness; Sales + Selling,1066,750,Qwen/Qwen2-1.5B-Instruct,0,0,0,0,0,0,Hospita in Enschede voor economisch daklozen,21 januari 2025 dinsdag 10:31 AM GMT,Social Security; Housing Market; Poverty + Homelessness; Sales + Selling
4,8291,Article ID: 8291\nDate: 20 maart 2025 donderdag 08:40 PM GMT\nTitle: De voorzitter zonder Elfstedentocht: 'Ook de tocht is niet eeuwigdurend'\nSubjects: Sports + Recreation E

### Check manual scores

Validation metrics are calculated only for rows where all six manual score columns have been filled in. Manual scores must use the same scale as the model output: `0`, `50`, or `100`.


In [ ]:
# Convert manual score columns to numeric values for validation checks.
for col in MANUAL_SCORE_COLS:
    validation_sample[col] = pd.to_numeric(validation_sample[col], errors="coerce")

# Keep only rows with complete manual scoring across all six news values.
complete_manual_df = validation_sample.dropna(subset=MANUAL_SCORE_COLS).copy()

print("Rows in validation sample:", len(validation_sample))
print("Rows with complete manual coding:", len(complete_manual_df))
print("Rows still missing manual coding:", len(validation_sample) - len(complete_manual_df))

# Confirm that completed manual scores use only the allowed values.
invalid_entries = []
for col in MANUAL_SCORE_COLS:
    invalid_mask = ~complete_manual_df[col].isin(ALLOWED_SCORES)
    if invalid_mask.any():
        invalid_entries.append((col, complete_manual_df.loc[invalid_mask, ["article_id", col]]))

if invalid_entries:
    for col, bad_rows in invalid_entries:
        print(f"Invalid values found in {col}:")
        display(bad_rows)
    raise ValueError("Manual scores must only be 0, 50, or 100.")

# Cast completed manual scores to integers before metric calculation.
for col in MANUAL_SCORE_COLS:
    complete_manual_df[col] = complete_manual_df[col].astype(int)

if complete_manual_df.empty:
    print("No complete manual coding yet. Fill the manual_*_score columns, save the CSV, and rerun this section.")
else:
    print("Manual validation scores look valid.")


Rows in validation sample: 300
Rows with complete manual coding: 300
Rows still missing manual coding: 0
Manual validation scores look valid.


### Define validation metrics

The following functions calculate agreement between manual scores and model scores. I used ChatGPT as coding support for the implementation of these metric functions and then applied them to the validation design used in this thesis.

The notebook calculates two types of agreement:

- exact three-level agreement for scores `0`, `50`, and `100`;
- present/absent agreement, where `50` and `100` both count as presence of a news value.


In [ ]:
# Calculate present/absent agreement metrics for one news value.
def binary_metrics(y_true, y_pred) -> dict:
    y_true = np.asarray(y_true).astype(bool)
    y_pred = np.asarray(y_pred).astype(bool)

    # Count true/false positives and negatives for binary presence.
    tp = int(np.sum(y_true & y_pred))
    tn = int(np.sum(~y_true & ~y_pred))
    fp = int(np.sum(~y_true & y_pred))
    fn = int(np.sum(y_true & ~y_pred))

    # Standard classification metrics.
    accuracy = (tp + tn) / len(y_true) if len(y_true) else np.nan
    precision = tp / (tp + fp) if (tp + fp) else np.nan
    recall = tp / (tp + fn) if (tp + fn) else np.nan
    f1 = 2 * precision * recall / (precision + recall) if pd.notna(precision) and pd.notna(recall) and (precision + recall) > 0 else np.nan

    return {
        "binary_accuracy": accuracy,
        "precision_present": precision,
        "recall_present": recall,
        "f1_present": f1,
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
    }


# Calculate exact three-level Cohen's kappa for scores 0, 50, and 100.
def cohen_kappa_score_simple(y_true, y_pred, labels=(0, 50, 100)) -> float:
    y_true = pd.Series(y_true)
    y_pred = pd.Series(y_pred)
    if len(y_true) == 0:
        return np.nan

    observed = np.mean(y_true.values == y_pred.values)
    expected = 0
    for label in labels:
        expected += np.mean(y_true.values == label) * np.mean(y_pred.values == label)

    if expected == 1:
        return np.nan
    return (observed - expected) / (1 - expected)


# Compare either calibrated or raw model scores with manual scores.
def evaluate_score_set(df: pd.DataFrame, score_set: str) -> pd.DataFrame:
    rows = []
    for value in NEWS_VALUE_NAMES:
        manual_col = f"manual_{value}_score"
        pred_col = f"{value}_score" if score_set == "calibrated" else f"raw_{value}_score"
        require_columns(df, [manual_col, pred_col], "validation dataframe")

        y_true = df[manual_col].astype(int)
        y_pred = df[pred_col].astype(int)

        # Treat scores 50 and 100 as present; treat score 100 as strong presence.
        present = binary_metrics(y_true > 0, y_pred > 0)
        strong = binary_metrics(y_true == 100, y_pred == 100)

        rows.append({
            "score_set": score_set,
            "news_value": value,
            "n": len(df),
            "exact_score_accuracy": float(np.mean(y_true == y_pred)),
            "mean_absolute_error": float(np.mean(np.abs(y_true - y_pred))),
            "cohen_kappa_3level": cohen_kappa_score_simple(y_true, y_pred),
            "manual_mean_score": float(y_true.mean()),
            "predicted_mean_score": float(y_pred.mean()),
            "manual_present_share": float((y_true > 0).mean()),
            "predicted_present_share": float((y_pred > 0).mean()),
            "manual_strong_share": float((y_true == 100).mean()),
            "predicted_strong_share": float((y_pred == 100).mean()),
            "present_accuracy": present["binary_accuracy"],
            "present_precision": present["precision_present"],
            "present_recall": present["recall_present"],
            "present_f1": present["f1_present"],
            "present_tp": present["tp"],
            "present_tn": present["tn"],
            "present_fp": present["fp"],
            "present_fn": present["fn"],
            "strong_accuracy": strong["binary_accuracy"],
            "strong_precision": strong["precision_present"],
            "strong_recall": strong["recall_present"],
            "strong_f1": strong["f1_present"],
            "strong_tp": strong["tp"],
            "strong_tn": strong["tn"],
            "strong_fp": strong["fp"],
            "strong_fn": strong["fn"],
        })

    return pd.DataFrame(rows)


### Calculate validation metrics

This cell saves `manual_validation_raw_vs_calibrated_metrics.csv`.

The table reports validation metrics separately for each news value and for both score sets: calibrated and raw Qwen2 scores.


In [ ]:
if complete_manual_df.empty:
    print("Validation metrics were not calculated because no rows have complete manual coding yet.")
else:
    # Calculate metrics for calibrated and raw score sets.
    validation_results = pd.concat(
        [
            evaluate_score_set(complete_manual_df, "calibrated"),
            evaluate_score_set(complete_manual_df, "raw_uncalibrated"),
        ],
        ignore_index=True,
    )

    # Save the detailed validation table for thesis reporting and later inspection.
    validation_metrics_path = output_path("manual_validation_raw_vs_calibrated_metrics.csv")
    validation_results.to_csv(validation_metrics_path, index=False, encoding="utf-8-sig")
    print("Saved:", validation_metrics_path)

    # Display the main agreement metrics in the notebook.
    display_cols = [
        "score_set",
        "news_value",
        "n",
        "exact_score_accuracy",
        "mean_absolute_error",
        "cohen_kappa_3level",
        "present_accuracy",
        "present_precision",
        "present_recall",
        "present_f1",
        "manual_present_share",
        "predicted_present_share",
    ]
    display(validation_results[display_cols].sort_values(["news_value", "score_set"]).round(3))


Saved: /content/drive/MyDrive/Thesis/data_processed/manual_validation_raw_vs_calibrated_metrics.csv


,score_set,news_value,n,exact_score_accuracy,mean_absolute_error,cohen_kappa_3level,present_accuracy,present_precision,present_recall,present_f1,manual_present_share,predicted_present_share
1,calibrated,bad_news,300,0.670,17.833,0.478,0.770,0.631,0.965,0.763,0.383,0.587
7,raw_uncalibrated,bad_news,300,0.750,14.500,0.504,0.837,0.902,0.643,0.751,0.383,0.273
4,calibrated,celebrity,300,0.833,8.500,0.527,0.860,0.815,0.579,0.677,0.253,0.180
10,raw_uncalibrated,celebrity,300,0.763,13.167,0.111,0.767,1.000,0.079,0.146,0.253,0.020
0,calibrated,entertainment,300,0.650,20.000,0.436,0.823,0.777,0.831,0.803,0.433,0.463
6,raw_uncalibrated,entertainment,300,0.677,27.167,0.300,0.680,0.972,0.269,0.422,0.433,0.120
3,calibrated,good_news,300,0.853,10.000,0.384,0.877,0.447,0.656,0.532,0.107,0.157
9,raw_uncalibrated,good_news,300,0.893,8.000,0.431,0.917,0.621,0.562,0.590,0.107,0.097
2,calibrated,magnitude,300,0.867,7.000,0.549,0.907,0.679,0.792,0.731,0.160,0.187
8,raw_uncalibrated,magnitude,300,0.757,12.500,0.391,0.797,0.436,0.917,0.591,0.160,0.337


### Compare calibrated and raw validation results

This cell creates two files:

- `manual_validation_raw_vs_calibrated_comparison.csv`;
- `manual_validation_article_level_disagreements.csv`.

The comparison file places raw and calibrated metrics side by side for each news value. The article-level disagreement file identifies cases where calibration reduces, increases, or does not change the absolute error compared with the manual scores.

I used ChatGPT as coding support for the reshaping and diagnostic-table implementation.


In [ ]:
if complete_manual_df.empty:
    print("Comparison and disagreement files were not created because no complete manual coding is available yet.")
else:
    # Put calibrated and raw metrics side by side.
    comparison = validation_results.pivot(
        index="news_value",
        columns="score_set",
        values=[
            "exact_score_accuracy",
            "mean_absolute_error",
            "cohen_kappa_3level",
            "present_f1",
            "present_precision",
            "present_recall",
            "manual_present_share",
            "predicted_present_share",
        ],
    )
    comparison.columns = [f"{metric}_{score_set}" for metric, score_set in comparison.columns]
    comparison = comparison.reset_index()
    # For accuracy/F1 differences, positive values indicate higher calibrated performance.
    # For MAE, negative values indicate lower calibrated error; positive values indicate higher calibrated error.
    comparison["exact_accuracy_difference_calibrated_minus_raw"] = comparison["exact_score_accuracy_calibrated"] - comparison["exact_score_accuracy_raw_uncalibrated"]
    comparison["present_f1_difference_calibrated_minus_raw"] = comparison["present_f1_calibrated"] - comparison["present_f1_raw_uncalibrated"]
    comparison["mae_difference_calibrated_minus_raw"] = comparison["mean_absolute_error_calibrated"] - comparison["mean_absolute_error_raw_uncalibrated"]

    comparison_path = output_path("manual_validation_raw_vs_calibrated_comparison.csv")
    comparison.to_csv(comparison_path, index=False, encoding="utf-8-sig")
    print("Saved:", comparison_path)
    display(comparison.round(3))

    # Create article-level diagnostics for manual inspection.
    diagnostic_df = complete_manual_df.copy()
    # For each news value, compare raw and calibrated scores with the manual score.
    for value in NEWS_VALUE_NAMES:
        manual_col = f"manual_{value}_score"
        raw_col = f"raw_{value}_score"
        calibrated_col = f"{value}_score"
        diagnostic_df[f"{value}_raw_matches_manual"] = diagnostic_df[raw_col].astype(int) == diagnostic_df[manual_col].astype(int)
        diagnostic_df[f"{value}_calibrated_matches_manual"] = diagnostic_df[calibrated_col].astype(int) == diagnostic_df[manual_col].astype(int)
        diagnostic_df[f"{value}_raw_absolute_error"] = (diagnostic_df[raw_col].astype(int) - diagnostic_df[manual_col].astype(int)).abs()
        diagnostic_df[f"{value}_calibrated_absolute_error"] = (diagnostic_df[calibrated_col].astype(int) - diagnostic_df[manual_col].astype(int)).abs()

    diagnostic_df["raw_total_absolute_error"] = sum(diagnostic_df[f"{value}_raw_absolute_error"] for value in NEWS_VALUE_NAMES)
    diagnostic_df["calibrated_total_absolute_error"] = sum(diagnostic_df[f"{value}_calibrated_absolute_error"] for value in NEWS_VALUE_NAMES)
    diagnostic_df["calibration_improved_total_error"] = diagnostic_df["calibrated_total_absolute_error"] < diagnostic_df["raw_total_absolute_error"]
    diagnostic_df["calibration_worsened_total_error"] = diagnostic_df["calibrated_total_absolute_error"] > diagnostic_df["raw_total_absolute_error"]

    # Save the article-level disagreement file.
    disagreement_path = output_path("manual_validation_article_level_disagreements.csv")
    diagnostic_df.to_csv(disagreement_path, index=False, encoding="utf-8-sig")
    print("Saved:", disagreement_path)
    print("Calibration improved total error for:", int(diagnostic_df["calibration_improved_total_error"].sum()), "articles")
    print("Calibration worsened total error for:", int(diagnostic_df["calibration_worsened_total_error"].sum()), "articles")
    print("Calibration made no difference for:", int((diagnostic_df["raw_total_absolute_error"] == diagnostic_df["calibrated_total_absolute_error"]).sum()), "articles")


Saved: /content/drive/MyDrive/Thesis/data_processed/manual_validation_raw_vs_calibrated_comparison.csv


,news_value,exact_score_accuracy_calibrated,exact_score_accuracy_raw_uncalibrated,mean_absolute_error_calibrated,mean_absolute_error_raw_uncalibrated,cohen_kappa_3level_calibrated,cohen_kappa_3level_raw_uncalibrated,present_f1_calibrated,present_f1_raw_uncalibrated,present_precision_calibrated,present_precision_raw_uncalibrated,present_recall_calibrated,present_recall_raw_uncalibrated,manual_present_share_calibrated,manual_present_share_raw_uncalibrated,predicted_present_share_calibrated,predicted_present_share_raw_uncalibrated,exact_accuracy_difference_calibrated_minus_raw,present_f1_difference_calibrated_minus_raw,mae_difference_calibrated_minus_raw
0,bad_news,0.670,0.750,17.833,14.500,0.478,0.504,0.763,0.751,0.631,0.902,0.965,0.643,0.383,0.383,0.587,0.273,-0.080,0.012,3.333
1,celebrity,0.833,0.763,8.500,13.167,0.527,0.111,0.677,0.146,0.815,1.000,0.579,0.079,0.253,0.253,0.180,0.020,0.070,0.531,-4.667
2,entertainment,0.650,0.677,20.000,27.167,0.436,0.300,0.803,0.422,0.777,0.972,0.831,0.269,0.433,0.433,0.463,0.120,-0.027,0.381,-7.167
3,good_news,0.853,0.893,10.000,8.000,0.384,0.431,0.532,0.590,0.447,0.621,0.656,0.562,0.107,0.107,0.157,0.097,-0.040,-0.059,2.000
4,magnitude,0.867,0.757,7.000,12.500,0.549,0.391,0.731,0.591,0.679,0.436,0.792,0.917,0.160,0.160,0.187,0.337,0.110,0.140,-5.500
5,power_elite,0.710,0.700,17.000,17.833,0.375,0.343,0.642,0.604,0.682,0.663,0.606,0.556,0.330,0.330,0.293,0.277,0.010,0.037,-0.833


Saved: /content/drive/MyDrive/Thesis/data_processed/manual_validation_article_level_disagreements.csv
Calibration improved total error for: 113 articles
Calibration worsened total error for: 53 articles
Calibration made no difference for: 134 articles


### Create confusion matrices

The confusion matrices show how often each manual score (`0`, `50`, or `100`) corresponds to each predicted score.


In [ ]:
if complete_manual_df.empty:
    print("Confusion matrices were not created because no complete manual coding is available yet.")
else:
    # Store all confusion matrices in long-table format for export.
    confusion_rows = []
    for score_set in ["calibrated", "raw_uncalibrated"]:
        for value in NEWS_VALUE_NAMES:
            manual_col = f"manual_{value}_score"
            pred_col = f"{value}_score" if score_set == "calibrated" else f"raw_{value}_score"
            # Cross-tabulate manual scores against predicted scores for each score set and news value.
            cm = pd.crosstab(
                complete_manual_df[manual_col].astype(int),
                complete_manual_df[pred_col].astype(int),
                rownames=["manual_score"],
                colnames=["predicted_score"],
                dropna=False,
            ).reindex(index=[0, 50, 100], columns=[0, 50, 100], fill_value=0)
            for manual_score in [0, 50, 100]:
                for predicted_score in [0, 50, 100]:
                    confusion_rows.append({
                        "score_set": score_set,
                        "news_value": value,
                        "manual_score": manual_score,
                        "predicted_score": predicted_score,
                        "count": int(cm.loc[manual_score, predicted_score]),
                    })

    # Save all confusion matrices in one CSV file.
    confusion_matrices = pd.DataFrame(confusion_rows)
    confusion_path = output_path("manual_validation_confusion_matrices_raw_vs_calibrated.csv")
    confusion_matrices.to_csv(confusion_path, index=False, encoding="utf-8-sig")
    print("Saved:", confusion_path)
    display(confusion_matrices.head(18))


Saved: /content/drive/MyDrive/Thesis/data_processed/manual_validation_confusion_matrices_raw_vs_calibrated.csv


,score_set,news_value,manual_score,predicted_score,count
0,calibrated,entertainment,0,0,139
1,calibrated,entertainment,0,50,30
2,calibrated,entertainment,0,100,1
3,calibrated,entertainment,50,0,8
4,calibrated,entertainment,50,50,22
5,calibrated,entertainment,50,100,1
6,calibrated,entertainment,100,0,14
7,calibrated,entertainment,100,50,51
8,calibrated,entertainment,100,100,34
9,calibrated,bad_news,0,0,120


## 7. Final summary checks

The final section creates summary tables for corpus-level score distributions and confirms which calibrated GIS files should be imported into GIS software.



### Summarize calibrated and raw article-level scores

This table summarizes the distribution of each news value across the full article corpus.

The output file is `corpus_score_summary_calibrated_and_raw.csv`.

In [ ]:
# Build a compact corpus-level summary for calibrated and raw score distributions.
corpus_rows = []
# Calculate distribution statistics for each news value and score set.
for score_set, cols in [("calibrated", SCORE_COLS), ("raw_uncalibrated", RAW_SCORE_COLS)]:
    for name, col in zip(NEWS_VALUE_NAMES, cols):
        scores = calibrated_articles[col].astype(float)
        corpus_rows.append({
            "score_set": score_set,
            "news_value": name,
            "n_articles": len(scores),
            "mean_score": scores.mean(),
            "present_count": int((scores > 0).sum()),
            "present_share": float((scores > 0).mean()),
            "strong_count": int((scores == 100).sum()),
            "strong_share": float((scores == 100).mean()),
            "score_0_count": int((scores == 0).sum()),
            "score_50_count": int((scores == 50).sum()),
            "score_100_count": int((scores == 100).sum()),
        })

# Save the corpus-level summary table.
corpus_score_summary = pd.DataFrame(corpus_rows)
corpus_summary_path = output_path("corpus_score_summary_calibrated_and_raw.csv")
corpus_score_summary.to_csv(corpus_summary_path, index=False, encoding="utf-8-sig")
print("Saved:", corpus_summary_path)
display(corpus_score_summary.round(3))


Saved: /content/drive/MyDrive/Thesis/data_processed/corpus_score_summary_calibrated_and_raw.csv


,score_set,news_value,n_articles,mean_score,present_count,present_share,strong_count,strong_share,score_0_count,score_50_count,score_100_count
0,calibrated,entertainment,12225,29.616,5708,0.467,1533,0.125,6517,4175,1533
1,calibrated,bad_news,12225,45.616,7600,0.622,3553,0.291,4625,4047,3553
2,calibrated,magnitude,12225,14.413,2837,0.232,687,0.056,9388,2150,687
3,calibrated,good_news,12225,15.542,2356,0.193,1444,0.118,9869,912,1444
4,calibrated,celebrity,12225,11.325,2493,0.204,276,0.023,9732,2217,276
5,calibrated,power_elite,12225,17.133,3714,0.304,475,0.039,8511,3239,475
6,raw_uncalibrated,entertainment,12225,12.589,1545,0.126,1533,0.125,10680,12,1533
7,raw_uncalibrated,bad_news,12225,29.063,3553,0.291,3553,0.291,8672,0,3553
8,raw_uncalibrated,magnitude,12225,21.088,4469,0.366,687,0.056,7756,3782,687
9,raw_uncalibrated,good_news,12225,12.037,1499,0.123,1444,0.118,10726,55,1444


### Show top locations and GIS import paths

This final check lists the most frequently mentioned locations and prints the two calibrated GIS files used outside the notebook.


In [ ]:
# Display the 25 locations with the highest number of article mentions.
top_locations = summary_by_location_calibrated.sort_values("mention_count", ascending=False).head(25)
display(top_locations[["location", "lat", "lon", "mention_count", "dominant_news_values", "dominant_map_category"]])

# Print the calibrated GIS mention and by-location summary files for import into GIS software.
print("Recommended GIS imports:")
print("1.", output_path("location_mentions_with_calibrated_news_values_for_gis.csv"))
print("2.", output_path("location_news_value_summary_by_location_calibrated.csv"))


,location,lat,lon,mention_count,dominant_news_values,dominant_map_category
27,Amsterdam,52.373080,4.892453,2335,bad_news,bad_news
632,Rotterdam,51.924442,4.477750,1482,bad_news,bad_news
161,Den Haag,52.079984,4.311346,1452,bad_news,bad_news
727,Utrecht,52.090701,5.121563,842,bad_news,bad_news
279,Groningen,53.219065,6.568008,651,bad_news,bad_news
217,Eindhoven,51.439265,5.478633,610,bad_news,bad_news
316,Heerenveen,52.956825,5.927005,472,entertainment,entertainment
40,Arnhem,51.984257,5.910857,390,bad_news,bad_news
709,Tilburg,51.556240,5.088601,282,bad_news,bad_news
116,Breda,51.588785,4.776024,281,bad_news,bad_news


Recommended GIS imports:
1. /content/drive/MyDrive/Thesis/data_processed/location_mentions_with_calibrated_news_values_for_gis.csv
2. /content/drive/MyDrive/Thesis/data_processed/location_news_value_summary_by_location_calibrated.csv
